## 15_model_task — the MODEL notebook of the task family (tasks 10-14 are the

table notebooks; this one adds model access through the SAME one access).
ONE ACCESS, TWO CATALOGS (C360-style; runbook Appendix A.5):
  - data catalog   : CFG.table("practice_customers_clean")
  - model registry : CFG.model("practice_linear_regression") -> models:/...@latest
Used by: practice-job-2-multitask (model_task, depends on task_c).
Writes: practice_customers_predictions (partitioned by ingest_date) — flows
through 35_export_tables -> 47_import_tables automatically (5th table).
PREREQ (old side): 01_train_linear_regression ran; (new side): models registered (50).
Widgets: SCHEMA/PARTITION/CONFIG_PATH. Tables: 2-part schema.table; models:
    <current_catalog>.default.<name> (Free Edition UC registry requirement).


In [ ]:

%run ../lib/config_lib


In [ ]:
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (OLD schema name — unchanged by migration)")
dbutils.widgets.text("PARTITION", "all", "all | latest | prev | YYYY-MM-DD")
dbutils.widgets.text("CONFIG_PATH", "", "Optional env config file (empty = built-in dev/stg/prd)")

CFG = setup_config(
    config_path=dbutils.widgets.get("CONFIG_PATH").strip(),
    run_date=dbutils.widgets.get("PARTITION").strip(),
    schema=dbutils.widgets.get("SCHEMA").strip(),
)


In [ ]:

import mlflow
import pandas as pd

src = CFG.table("practice_customers_clean")
frame = features_from_df(CFG.partition_filter(spark.read.table(src), table=src))

MODEL_NAME = CFG.model("practice_linear_regression")
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@latest")
print(f"model  : {MODEL_NAME}@latest")
print(f"source : {src} (PARTITION={CFG.run_date})")

preds = model.predict(frame[FEATURES].to_numpy(dtype=float))

out_pdf = frame[["customer_id", "ingest_date"]].copy()
out_pdf["predicted_revenue"] = preds
out = spark.createDataFrame(out_pdf)
CFG.write_partitioned(out, "practice_customers_predictions")

dst = CFG.table("practice_customers_predictions")
n = spark.read.table(dst).count()
print(f"MODEL TASK DONE — scored {len(out_pdf)} rows with {MODEL_NAME}@latest -> {dst} ({n} rows in table)")
